# Swell Models


In [1]:
import json
import os
import pickle
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    fbeta_score,
    log_loss,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from xgboost import XGBClassifier


In [2]:
path = kagglehub.dataset_download("qiriro/swell-heart-rate-variability-hrv")
train = pd.read_csv(os.path.join(path, "hrv dataset", "data", "final", "train.csv"))
test = pd.read_csv(os.path.join(path, "hrv dataset", "data", "final", "test.csv"))

print("Dataset path:", path)
print("Train shape:", train.shape)
print("Test shape:", test.shape)
print(train["condition"].value_counts())
print(test["condition"].value_counts())


Dataset path: C:\Users\tenuk\.cache\kagglehub\datasets\qiriro\swell-heart-rate-variability-hrv\versions\1
Train shape: (369289, 36)
Test shape: (41033, 36)
condition
no stress        200082
interruption     105150
time pressure     64057
Name: count, dtype: int64
condition
no stress        22158
interruption     11782
time pressure     7093
Name: count, dtype: int64


In [3]:
drop_columns = [
    "datasetId",
    "VLF",
    "LF",
    "HF",
    "TP",
    "LF_HF",
    "HF_LF",
    "LF_NU",
    "HF_NU",
    "LF_PCT",
    "HF_PCT",
]
window_size = 100
max_sequences_factor = 50
feature_columns = [column for column in train.columns if column not in drop_columns + ["condition"]]

def make_sequences(frame, window_size, feature_cols, max_windows=None):
    features = frame[feature_cols].to_numpy(dtype=np.float32)
    labels = frame["condition"].to_numpy()
    window_features = []
    window_labels = []
    window_indices = []

    for start in range(0, len(frame) - window_size + 1):
        end = start + window_size
        window_features.append(features[start:end].reshape(-1))
        window_labels.append(labels[end - 1])
        window_indices.append(end - 1)

    if max_windows is not None and len(window_features) > max_windows:
        indices = np.linspace(0, len(window_features) - 1, max_windows, dtype=int)
        window_features = [window_features[index] for index in indices]
        window_labels = [window_labels[index] for index in indices]
        window_indices = [window_indices[index] for index in indices]

    return (
        np.asarray(window_features, dtype=np.float32),
        np.asarray(window_labels),
        np.asarray(window_indices),
    )

max_windows = window_size * max_sequences_factor
train_X, train_y_raw, train_window_indices = make_sequences(train, window_size, feature_columns, max_windows)
test_X, test_y_raw, test_window_indices = make_sequences(test, window_size, feature_columns, max_windows)

print("Window size:", window_size)
print("Train X shape:", train_X.shape)
print("Train y distribution:")
print(pd.Series(train_y_raw).value_counts().sort_index())
print("Test X shape:", test_X.shape)
print("Test y distribution:")
print(pd.Series(test_y_raw).value_counts().sort_index())


Window size: 100
Train X shape: (5000, 2400)
Train y distribution:
interruption     1422
no stress        2745
time pressure     833
Name: count, dtype: int64
Test X shape: (5000, 2400)
Test y distribution:
interruption     1442
no stress        2678
time pressure     880
Name: count, dtype: int64


In [ ]:
artifact_dir = Path(os.path.join("..","..","artifacts","swell","probability-classifier"))
artifact_dir.mkdir(parents=True, exist_ok=True)

le = LabelEncoder()
le.fit(train_y_raw)

train_X_fit, val_X, train_y_fit_raw, val_y_raw = train_test_split(
    train_X,
    train_y_raw,
    test_size=0.2,
    random_state=42,
    stratify=train_y_raw,
)

scaler = StandardScaler()
train_X_fit = scaler.fit_transform(train_X_fit)
val_X = scaler.transform(val_X)
test_X_scaled = scaler.transform(test_X)

train_y = le.transform(train_y_fit_raw)
val_y = le.transform(val_y_raw)
test_y = le.transform(test_y_raw)

class_counts = np.bincount(train_y)
class_weight = {i: len(train_y) / (len(le.classes_) * count) for i, count in enumerate(class_counts)}
sample_weight = np.array([class_weight[label] for label in train_y])

model = XGBClassifier(
    n_estimators=4000,
    max_depth=8,
    learning_rate=0.01,
    subsample=0.85,
    colsample_bytree=0.75,
    min_child_weight=2,
    gamma=0.0,
    reg_alpha=0.0,
    reg_lambda=2.0,
    objective="multi:softprob",
    eval_metric="mlogloss",
    early_stopping_rounds=20,
    random_state=42,
)

model.fit(train_X_fit, train_y, sample_weight=sample_weight, eval_set=[(val_X, val_y)], verbose=False)

window_artifact_dir = artifact_dir / f"window_{window_size}"
window_artifact_dir.mkdir(parents=True, exist_ok=True)

with open(window_artifact_dir / "model.pkl", "wb") as handle:
    pickle.dump(model, handle)
with open(window_artifact_dir / "scaler.pkl", "wb") as handle:
    pickle.dump(scaler, handle)

with open(artifact_dir / "label_encoder.pkl", "wb") as handle:
    pickle.dump(le, handle)
with open(artifact_dir / "metadata.json", "w", encoding="utf-8") as handle:
    json.dump(
        {
            "window_size": window_size,
            "max_sequences_factor": max_sequences_factor,
            "feature_columns": feature_columns,
            "classes": le.classes_.tolist(),
        },
        handle,
        indent=2,
    )

print("Saved window", window_size, "to:", window_artifact_dir.resolve())
print("Best iteration:", getattr(model, "best_iteration", None))
print("Best score:", getattr(model, "best_score", None))


In [ ]:
val_prob = model.predict_proba(val_X)
test_prob = model.predict_proba(test_X_scaled)
no_stress_index = int(np.where(le.classes_ == "no stress")[0][0])

def predict_with_threshold(probabilities, threshold):
    predictions = np.argmax(probabilities, axis=1)
    no_stress_mask = probabilities[:, no_stress_index] >= threshold
    predictions[no_stress_mask] = no_stress_index
    return predictions

threshold_rows = []
val_stress_y = (val_y != no_stress_index).astype(int)
for threshold in np.linspace(0.3, 0.8, 26):
    val_pred = predict_with_threshold(val_prob, float(threshold))
    val_stress_pred = (val_pred != no_stress_index).astype(int)
    threshold_rows.append(
        {
            "threshold": float(threshold),
            "stress_recall": recall_score(val_stress_y, val_stress_pred, zero_division=0),
            "stress_precision": precision_score(val_stress_y, val_stress_pred, zero_division=0),
            "stress_f2": fbeta_score(val_stress_y, val_stress_pred, beta=2, zero_division=0),
        }
    )

threshold_df = pd.DataFrame(threshold_rows).sort_values(["stress_f2", "stress_recall", "stress_precision"], ascending=False)
best_threshold = float(threshold_df.iloc[0]["threshold"])
print(f"Best stress threshold: {best_threshold:.2f}")
print(threshold_df.head(5).to_string(index=False))

y_pred = np.argmax(test_prob, axis=1)
tuned_pred = predict_with_threshold(test_prob, best_threshold)
cm = confusion_matrix(test_y, y_pred, labels=np.arange(len(le.classes_)))
tuned_cm = confusion_matrix(test_y, tuned_pred, labels=np.arange(len(le.classes_)))

def summarize(cm):
    tp = np.diag(cm)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp
    return pd.DataFrame({
        "class": le.classes_,
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
    })

stress_test_y = (test_y != no_stress_index).astype(int)
stress_test_pred = (tuned_pred != no_stress_index).astype(int)

print("Baseline confusion matrix:\n", cm)
print(summarize(cm).to_string(index=False))
print(classification_report(test_y, y_pred, target_names=le.classes_))
print("Tuned confusion matrix:\n", tuned_cm)
print(summarize(tuned_cm).to_string(index=False))
print(classification_report(test_y, tuned_pred, target_names=le.classes_))
print(
    pd.DataFrame(
        [{
            "window_size": window_size,
            "accuracy": accuracy_score(test_y, tuned_pred),
            "macro_f1": classification_report(test_y, tuned_pred, target_names=le.classes_, output_dict=True)["macro avg"]["f1-score"],
            "stress_precision": precision_score(stress_test_y, stress_test_pred, zero_division=0),
            "stress_recall": recall_score(stress_test_y, stress_test_pred, zero_division=0),
            "stress_f2": fbeta_score(stress_test_y, stress_test_pred, beta=2, zero_division=0),
            "log_loss": log_loss(test_y, test_prob),
        }]
    ).to_string(index=False)
)
